# 04 — Patient Pathway Analysis
**CareFlow Process Mining | Google Colab**

Uses `case_id`, `activity`, `event_index`, and `timestamp` to construct recorded activity sequences per case. This describes the sequence in the event log; it does not establish clinical causality or whether care was appropriate.

In [ ]:
!pip -q install openpyxl pandas numpy matplotlib seaborn pm4py

import io, os, re, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("CareFlow Process Mining environment ready.")

In [ ]:
# Upload: Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the processed sepsis workbook.")
workbook_name = next(iter(uploaded))
workbook_bytes = uploaded[workbook_name]
xls = pd.ExcelFile(io.BytesIO(workbook_bytes))
print("Workbook:", workbook_name)
print("Sheets:", xls.sheet_names)

required_sheet = "Event Log Visual"
if required_sheet not in xls.sheet_names:
    raise ValueError(f"Expected sheet '{required_sheet}'. Available sheets: {xls.sheet_names}")

events = pd.read_excel(io.BytesIO(workbook_bytes), sheet_name="Event Log Visual")
events.columns = [str(c).strip() for c in events.columns]
print("Event Log Visual shape:", events.shape)
display(events.head())

In [ ]:
# This workbook's event-log sheet uses these verified column names:
CASE_COL = "case_id"
ACTIVITY_COL = "activity"
TIME_COL = "timestamp"
EVENT_INDEX_COL = "event_index"
ORG_COL = "org_group"
LIFECYCLE_COL = "lifecycle_transition"

# Parse timestamp safely; invalid values become NaT and are reported.
if TIME_COL in events.columns:
    events[TIME_COL] = pd.to_datetime(events[TIME_COL], errors="coerce")

print("Case column:", CASE_COL, "| present:", CASE_COL in events.columns)
print("Activity column:", ACTIVITY_COL, "| present:", ACTIVITY_COL in events.columns)
print("Timestamp column:", TIME_COL, "| present:", TIME_COL in events.columns)

## 1. Prepare event order

In [ ]:
path = events.dropna(subset=[CASE_COL, ACTIVITY_COL]).copy()
path[TIME_COL] = pd.to_datetime(path[TIME_COL], errors="coerce")
order_cols = [CASE_COL]
if EVENT_INDEX_COL in path.columns:
    order_cols.append(EVENT_INDEX_COL)
order_cols.append(TIME_COL)
path = path.sort_values(order_cols, na_position="last")
display(path[[CASE_COL, EVENT_INDEX_COL, ACTIVITY_COL, TIME_COL]].head(12))

## 2. Build sequences per case

In [ ]:
case_sequences = (
    path.groupby(CASE_COL)[ACTIVITY_COL]
    .apply(lambda values: " → ".join(values.astype(str)))
    .rename("activity_sequence")
    .reset_index()
)
display(case_sequences.head(10))

## 3. Most common pathways

In [ ]:
top_paths = case_sequences["activity_sequence"].value_counts().head(15)
display(top_paths.rename_axis("activity_sequence").reset_index(name="case_count"))
plt.figure(figsize=(12, 7))
top_paths.sort_values().plot(kind="barh")
plt.title("Top 15 Recorded Activity Sequences")
plt.xlabel("Number of cases")
plt.ylabel("Activity sequence")
plt.tight_layout()
plt.show()

## 4. Directly-follows activity pairs

In [ ]:
path["_next_activity"] = path.groupby(CASE_COL)[ACTIVITY_COL].shift(-1)
pairs = path.dropna(subset=["_next_activity"])
directly_follows = (
    pairs.groupby([ACTIVITY_COL, "_next_activity"])
    .size().sort_values(ascending=False).head(30)
    .rename("count").reset_index()
)
directly_follows.columns = ["activity", "next_activity", "count"]
display(directly_follows)

## 5. Export results

In [ ]:
case_sequences.to_csv("patient_pathway_sequences.csv", index=False)
directly_follows.to_csv("directly_follows_pairs.csv", index=False)
files.download("patient_pathway_sequences.csv")
files.download("directly_follows_pairs.csv")

**Privacy note:** Do not publish case-level sequences if they could expose sensitive patient information. Aggregate results are safer for public reports.